Official Doc:
https://learn.microsoft.com/en-us/azure/databricks/data-governance/unity-catalog/abac/tutorial

In [0]:
%sql
USE CATALOG azuredatabricks_catalog;

USE SCHEMA gold;

CREATE TABLE IF NOT EXISTS profiles (
    First_Name STRING,
    Last_Name STRING,
    Phone_Number STRING,
    Address STRING,
    SSN STRING
);

INSERT INTO profiles (First_Name, Last_Name, Phone_Number, Address, SSN)
VALUES
('John', 'Doe', '123-456-7890', '123 Main St, NY', '123-45-6789'),
('Jane', 'Smith', '234-567-8901', '456 Oak St, CA', '234-56-7890'),
('Alice', 'Johnson', '345-678-9012', '789 Pine St, TX', '345-67-8901'),
('Bob', 'Brown', '456-789-0123', '321 Maple St, FL', '456-78-9012'),
('Charlie', 'Davis', '567-890-1234', '654 Cedar St, IL', '567-89-0123'),
('Emily', 'White', '678-901-2345', '987 Birch St, WA', '678-90-1234'),
('Frank', 'Miller', '789-012-3456', '741 Spruce St, WA', '789-01-2345'),
('Grace', 'Wilson', '890-123-4567', '852 Elm St, NV', '890-12-3456'),
('Hank', 'Moore', '901-234-5678', '963 Walnut St, CO', '901-23-4567'),
('Ivy', 'Taylor', '012-345-6789', '159 Aspen St, AZ', '012-34-5678'),
('Liam', 'Connor', '111-222-3333', '12 Abbey Street, Dublin, Ireland EU', '111-22-3333'),
('Sophie', 'Dubois', '222-333-4444', '45 Rue de Rivoli, Paris, France Europe', '222-33-4444'),
('Hans', 'Müller', '333-444-5555', '78 Berliner Str., Berlin, Germany E.U.', '333-44-5555'),
('Elena', 'Rossi', '444-555-6666', '23 Via Roma, Milan, Italy Europe', '444-55-6666'),
('Johan', 'Andersson', '555-666-7777', '56 Drottninggatan, Stockholm, Sweden EU', '555-66-7777');


In [0]:
%sql
select * from profiles

# Row Level

In [0]:
%sql
CREATE OR REPLACE FUNCTION is_not_eu_address(address STRING)
RETURNS BOOLEAN
RETURN 
(
    SELECT 
        CASE 
        WHEN LOWER(address) LIKE '%eu%'
          OR LOWER(address) LIKE '%e.u.%'
          OR LOWER(address) LIKE '%europe%'
        THEN FALSE 
        ELSE TRUE 
        END
);

In [0]:
%sql
ALTER TABLE profiles
ALTER COLUMN address 
SET TAGS ("pii_tag"="address");

# Create Policy

In [0]:
%sql
CREATE OR REPLACE POLICY `eu_policy_not`
ON SCHEMA `azuredatabricks_catalog`.`gold`
ROW FILTER `azuredatabricks_catalog`.`gold`.`is_not_eu_address`
TO `account users`
EXCEPT `Admins`
FOR TABLES
MATCH COLUMNS has_tag_value('pii_tag','address') AS u0
USING COLUMNS (u0)

In [0]:
%sql
SELECT * FROM profiles

# Column Level

In [0]:
%sql
CREATE OR REPLACE FUNCTION mask_ssn(ssn STRING)
RETURN "******";

In [0]:
%sql
ALTER TABLE profiles
ALTER COLUMN ssn 
SET TAGS ("pii_tag"="ssn");

In [0]:
%sql
CREATE OR REPLACE POLICY `mask_ssn_policy`
ON CATALOG `azuredatabricks_catalog`
COLUMN MASK `azuredatabricks_catalog`.`gold`.`mask_ssn`
TO `account users`
EXCEPT `Admins`
FOR TABLES
MATCH COLUMNS has_tag_value('pii_tag','ssn') AS m
ON COLUMN m


In [0]:
%sql
SELECT * FROM profiles

In [0]:
%sql
ALTER TABLE profiles
ALTER COLUMN ssn 
UNSET TAGS ("pii_tag");

In [0]:
%sql
ALTER TABLE profiles
ALTER COLUMN address 
UNSET TAGS ("pii_tag");